# 第 18 章互动实验：金模板、阈值与过检漏检

用合成的周期版图和注入缺陷，观察阈值 k、统计方法对检出和误报的影响。

In [ ]:
%matplotlib inline
import importlib.util, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Noto Sans CJK SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

CHAPTER_NAME = "18_传统缺陷检测"
ASSIGNMENT = "18-1_金模板比对.py"

def find_chapter(name):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == name:
            return p
        if (p / name).is_dir():
            return p / name
    raise FileNotFoundError("请在课程目录内启动 Jupyter")

CHAPTER = find_chapter(CHAPTER_NAME)
COURSE_ROOT = CHAPTER.parent

def load_module(chapter_dir, filename):
    override = os.environ.get("COURSE_IMPL_DIR")
    base = Path(override) / chapter_dir.name if override else chapter_dir
    spec = importlib.util.spec_from_file_location(Path(filename).stem, base / filename)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

mod = load_module(CHAPTER, ASSIGNMENT)

def attempt(fn, *args, **kwargs):
    """调用作业函数；还没实现时打印提示并返回 None，而不是中断整个 Notebook。"""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f"先完成作业里的 {fn.__name__}：{exc}")
        return None

try:
    import cv2  # 可选，用于和 OpenCV 对照
except ImportError:
    cv2 = None
print("作业文件：", CHAPTER / ASSIGNMENT, "｜OpenCV：", "可用" if cv2 else "未安装（对照实验会跳过）")


In [ ]:
def pattern(h=96, w=96):
    y, x = np.mgrid[0:h, 0:w]
    return 40.0 + ((x // 4) % 2) * 80.0 + (((y // 16) % 2 == 0) & ((x // 16) % 2 == 0)) * 60.0

def scene(seed, n_goods=12, noise=3.0):
    rng = np.random.default_rng(seed)
    base = pattern()
    goods = base + rng.normal(scale=noise, size=(n_goods, 96, 96))
    test = base + rng.normal(scale=noise, size=(96, 96))
    truth = []
    for _ in range(5):
        cy, cx = rng.integers(8, 88, 2)
        amp = rng.choice([-1, 1]) * rng.uniform(12, 40)
        test[cy - 1:cy + 2, cx - 1:cx + 2] += amp
        truth.append((float(cy), float(cx)))
    return goods, test, truth

goods, test, truth = scene(0)
res = attempt(mod.robust_golden, goods)
if res is not None:
    median, sigma = res
    z = np.abs(test - median) / sigma
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    ax[0].imshow(test, cmap="gray"); ax[0].set_title("待检图")
    ax[1].imshow(sigma, cmap="viridis"); ax[1].set_title("逐像素 sigma")
    ax[2].imshow(z, cmap="magma", vmax=10); ax[2].set_title("z 图")
    for cy, cx in truth: ax[2].plot(cx, cy, "c+", ms=12)
    plt.show()

## 实验 1：k 与检出、误报

预测：k 从 3 增加到 9，检出率和误报数各怎么变？曲线的拐点在哪？

In [ ]:
ks = [3, 4, 5, 6, 7, 8, 9]
cap, fp = [], []
for k in ks:
    tp_all = n_all = fp_all = 0
    for s in range(10):
        g, t, tr = scene(100 + s)
        r = attempt(mod.robust_golden, g)
        if r is None: break
        found = attempt(mod.detect_defects, t, *r, k=k, min_area=3)
        if found is None: break
        m = attempt(mod.match_defects, [d["centroid"] for d in found], tr, 3.0)
        if m is None: break
        tp_all += m["tp"]; n_all += len(tr); fp_all += m["fp"]
    else:
        cap.append(tp_all / n_all); fp.append(fp_all / 10); continue
    break
if len(cap) == len(ks):
    fig, ax1 = plt.subplots(); ax2 = ax1.twinx()
    ax1.plot(ks, cap, "o-b"); ax1.set_ylabel("检出率", color="b")
    ax2.plot(ks, fp, "s-r"); ax2.set_ylabel("每张图误报", color="r")
    ax1.set_xlabel("k"); plt.title("阈值与检出、误报"); plt.show()

## 实验 2：良品里混进坏图

把 3 张良品替换成带大缺陷的图，比较中位数加 MAD 和均值加标准差。

In [ ]:
g, t, tr = scene(7)
g = g.copy()
g[:3, 40:56, 40:56] += 80
r = attempt(mod.robust_golden, g)
if r is not None:
    med, sig = r
    mean, std = g.mean(0), np.maximum(g.std(0), 1.0)
    for name, (c, s) in {"中位数+MAD": (med, sig), "均值+标准差": (mean, std)}.items():
        found = mod.detect_defects(t, c, s, k=6, min_area=3)
        m = mod.match_defects([d["centroid"] for d in found], tr, 3.0)
        print(f"{name}：检出 {m['tp']}/{len(tr)}，误报 {m['fp']}，污染区 sigma 均值 {s[40:56, 40:56].mean():.1f}")

## 破坏实验与记录

- 让待检图整体亮 10%，误报会发生什么？怎样修？
- 把 `min_area` 设为 1 和 9，分别会丢掉什么？
- 把实验 1 的曲线和你选的工作点写进 `实验记录.md`。